# LLaMA Token-Generation Latency Benchmark — GPU (Colab)

This notebook runs all benchmark experiments on a Colab GPU runtime.

**Setup:**
1. Runtime → Change runtime type → **GPU** (T4 or A100 with Colab Pro)
2. Set your GitHub repo URL in the next cell
3. Run cells top-to-bottom
4. Download results at the end

In [ ]:
# ============================================================
# Clone project from GitHub
# Set your repo URL below, then run this cell
# ============================================================
REPO_URL = "https://github.com/YOUR_USERNAME/Adv-Architecture-Research-Project.git"

!git clone {REPO_URL}
%cd Adv-Architecture-Research-Project

# Verify we're in the right directory
!ls -la

In [ ]:
# ============================================================
# Cell 2: HuggingFace Login, Install Dependencies & Verify GPU
# ============================================================
!nvidia-smi

# Login to HuggingFace (required for gated meta-llama models)
!pip install -q huggingface-hub
from huggingface_hub import login
login()

# Install llama-cpp-python with CUDA support
!pip install llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122

# Install other dependencies
!pip install transformers accelerate
!pip install numpy pandas matplotlib seaborn scipy

# Verify CUDA is available
import torch
print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_mem // (1024**2)} MB")

In [ ]:
# ============================================================
# Cell 3: Download GGUF Models
# ============================================================
!bash scripts/download_models.sh

# Verify models downloaded
!ls -lh models/*.gguf

In [ ]:
# ============================================================
# Cell 4: Quick Smoke Test (1 trial, 32 tokens)
# ============================================================
!python -m benchmarks.benchmark_harness \
    --model llama-3.2-1b-q4 \
    --prompt-length 128 \
    --trials 1 \
    --warmup 1 \
    --output-tokens 32 \
    --output smoke_test.csv

In [ ]:
# ============================================================
# Cell 5: Goal 1 — Full Benchmark Harness
# ============================================================
# Runs Q4_K_M across all prompt lengths (128, 256, 512, 1024)
!python -m benchmarks.benchmark_harness \
    --model llama-3.2-1b-q4 \
    --trials 10 \
    --output benchmark_results.csv

In [ ]:
# ============================================================
# Cell 6: Goal 2 — Latency Decomposition (meta-llama/Llama-3.2-1B-Instruct + CUDA)
# ============================================================
!python -m benchmarks.latency_decomposition \
    --device cuda \
    --tokens 64 \
    --output decomposition_results.json

In [ ]:
# ============================================================
# Cell 7: Goal 3 — Scaling Analysis (all 3 experiments)
# ============================================================
!python -m benchmarks.scaling_analysis \
    --experiment all \
    --trials 5 \
    --output scaling_results

In [ ]:
# ============================================================
# Cell 8: Goal 4 — Bottleneck Analysis
# ============================================================
!python -m analysis.bottleneck_analysis --model 1B --quant Q4_K_M

In [ ]:
# ============================================================
# Cell 9: Generate Plots
# ============================================================
!python -m analysis.plot_results

# Display generated plots inline
import os
import glob
from IPython.display import Image, display

for img_path in sorted(glob.glob('results/*.png')):
    print(f"\n--- {os.path.basename(img_path)} ---")
    display(Image(filename=img_path, width=800))

In [ ]:
# ============================================================
# Cell 10: Download Results
# ============================================================
import shutil
import os

# Zip data and results
shutil.make_archive('gpu_benchmark_results', 'zip', '.', 'data')
shutil.make_archive('gpu_benchmark_plots', 'zip', '.', 'results')

# List what's being downloaded
print("Raw data files:")
for f in sorted(glob.glob('data/raw/*')):
    size_kb = os.path.getsize(f) / 1024
    print(f"  {f} ({size_kb:.1f} KB)")

print("\nPlot files:")
for f in sorted(glob.glob('results/*.png')):
    size_kb = os.path.getsize(f) / 1024
    print(f"  {f} ({size_kb:.1f} KB)")

# Download (works in Colab browser)
try:
    from google.colab import files
    files.download('gpu_benchmark_results.zip')
    files.download('gpu_benchmark_plots.zip')
    print("\nDownload started!")
except ImportError:
    print("\nNot running in Colab browser — copy files manually from:")
    print(f"  {os.path.abspath('gpu_benchmark_results.zip')}")
    print(f"  {os.path.abspath('gpu_benchmark_plots.zip')}")